Helpful links:
- https://docs.llamaindex.ai/en/stable/getting_started/concepts/
- https://docs.llamaindex.ai/en/stable/understanding/
- https://docs.llamaindex.ai/en/stable/optimizing/production_rag/

In [ ]:
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex
from llama_index.core.indices.vector_store import VectorIndexRetriever
from llama_index.core.vector_stores import VectorStoreQuery
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.litellm import LiteLLM

from src.food_rag.utils import embedding_helper, hummus_docify, postgres_helper
from src.utils import load_util

In [ ]:
## Output Paths
data_path = './data'
embedding_path = data_path + '/embeddings/snowflake_l'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

In [ ]:
# Init LLM
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1"
)

embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# 1. Setup

### Steps:
1. Load HUMMUS recipe data
2. Load embedding models / sentence transformers
3. Load LLM (Llama 2)
4. Init Vector store (postgres)

## 1.1 Load data & preprocess data

In [ ]:
# Load data
hummus_data = load_util.get_current_hummus(data_path)

## 1.2 Initialize & connect to Postgres

Setup postgres on your local system.

Windows:
1. Use & install [docker desktop](https://www.docker.com/blog/how-to-use-the-postgres-docker-official-image/).
2. Download pgvector including postgres `docker pull pgvector/pgvector:pg16`
3. a) Run the container: `docker run --name rag_postgres -e POSTGRES_USER=postgres -e POSTGRES_PASSWORD=postgres -p 5432:5432 -d pgvector/pgvector:pg16`

   b) Or restart the container: `docker start rag_postgres`

Linux (Refer to this [GitLab page](https://github.com/pgvector/pgvector)):
1. Install pgvector with conda: `conda install -c conda-forge pgvector`
2. Install postgres: `sudo apt install postgresql postgresql-contrib`
3. Start postgres: `sudo systemctl start postgresql` and `sudo systemctl enable postgresql`
4. Set postgres default user: `sudo -i -u postgres` and `psql` and `ALTER USER postgres WITH PASSWORD 'postgres';` and `ALTER USER postgres WITH PASSWORD 'postgres';`, and finally switch back to your user with `exit`
5. Install pgvector via postgres repo: `sudo apt install -y postgresql-common` and `sudo /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh` and `sudo apt install postgresql-14-pgvector` while replacing the 14 with your postgres version


In [ ]:
# Postgres credentials
postgres_credentials = {
    'db_name': 'rag_postgres',
    'host': 'localhost',
    'password': 'postgres',
    'port': '5432',
    'user': 'postgres',
}

In [ ]:
# Connect to the default 'postgres' database
vector_store = postgres_helper.PostgresHelper(postgres_credentials, reset=True).connect(table_name='hummus_recipes',
                                                                                        embed_dim=1024)

# 2. Build Docs & put them in the Vector Store
## 2.1 Load Docs with embeddings

In [ ]:
# Load embeddings
recipe_documents = hummus_docify.recipes_to_docs(hummus_data['recipes_df'])
doc_nodes_dict, loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/recipe_documents')

In [ ]:
# Check if all recipe documents are loaded
embedding_helper.check_batches(recipe_documents, doc_nodes_dict)

In [ ]:
# Extract nodes from the loaded batches
recipe_nodes = [node for node_list in doc_nodes_dict.values() for node in node_list]

## 2.2 Put Nodes into the Vector Store
We now insert these nodes into our `PostgresVectorStore`.


In [ ]:
vector_store_index = VectorStoreIndex.from_vector_store(vector_store)
_ = vector_store_index.build_index_from_nodes(nodes=recipe_nodes)

# 3. Build Retrieval Pipeline

In [ ]:
query_str = "Can you recommend me a healthy hummus recipe?"

## a) Manually with queries

In [ ]:
# Get a Query Embedding
query_embedding = embed_model.get_query_embedding(query_str)

In [ ]:
# construct vector store query
query_mode = "default"
# query_mode = "sparse"
# query_mode = "hybrid"
# query_mode = "text_search"

vector_store_query = VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=2, mode=query_mode)

# returns a VectorStoreQueryResult
query_result = vector_store.query(vector_store_query)
print("First result:\n\n" + query_result.nodes[0].get_metadata_str())

## b) Use a Retriever

In [ ]:
# Initialize the retriever
query_mode = "default"  # =^= (dense)
# query_mode = "sparse"
# query_mode = "hybrid"
# query_mode = "text_search"

retriever = VectorIndexRetriever(index=vector_store_index, vector_store_query_mode=query_mode, similarity_top_k=10)

# Test the retriever
retrieved_docs = retriever.retrieve(query_str)
print("First result:\n\n" + retrieved_docs[0].get_content(metadata_mode="all"))